In [0]:
# 1. Setup paths
source_path = "s3://databricksawsara/"
checkpoint_path = "/Volumes/workspace/default/checkpoints/auto_loader/raw_data"
schema_path = "/Volumes/workspace/default/schemas/auto_loader/raw_data"

# 2. Configure the Auto Loader stream
df = (spark.readStream
  .format("cloudFiles")
  .option("cloudFiles.format", "json")
  .option("cloudFiles.schemaLocation", schema_path)
  .option("cloudFiles.inferColumnTypes", "true") # Optional: forceful typing
  .load(source_path))

# 3. Write to a Delta Table
query = (df.writeStream
  .option("checkpointLocation", checkpoint_path)
  .option("mergeSchema", "true") # Handles Schema Evolution
  .trigger(availableNow=True)    # Process all new data and stop (saves money!)
  .table("bronze_raw_data"))

In [0]:
%sql
-- Create a table that automatically updates when new files arrive
CREATE OR REFRESH STREAMING TABLE bronze_raw_data1
AS SELECT * FROM STREAM read_files(
  "s3://databricksawsara/",
  format => "json",
  header => "true",
  inferSchema => "true"
);

In [0]:
%sql
-- Bronze Layer: Ingesting raw data
CREATE OR REFRESH STREAMING TABLE bronze_orders
AS SELECT * FROM STREAM read_files("s3://databricksawsara/order", format => "json");


In [0]:
%sql
CREATE OR REFRESH STREAMING TABLE silver_orders_cleaned (
  -- Rule 1: Amount must be positive. If not, DROP the row.
  CONSTRAINT valid_amount EXPECT (amount > 0) ON VIOLATION DROP ROW,
  
  -- Rule 2: Order ID must not be null. If null, FAIL the whole job.
  CONSTRAINT mandatory_id EXPECT (order_id IS NOT NULL) ON VIOLATION FAIL UPDATE
)
AS SELECT * FROM STREAM(LIVE.bronze_orders);

In [0]:
%python
import dlt
from pyspark.sql.functions import col

@dlt.table
@dlt.expect_or_drop("valid_amount", "amount > 0")
@dlt.expect_or_fail("mandatory_id", "order_id IS NOT NULL")
def silver_orders_cleaned():
    return dlt.read_stream("bronze_orders")

In [0]:
source_path = "s3://databricksawsara/"
checkpoint_path = "/Volumes/workspace/default/checkpoints/auto_loader/raw_data"

df = (spark.readStream
  .format("cloudFiles")
  .option("cloudFiles.format", "json")
  # This is the magic option:
  .option("cloudFiles.schemaEvolutionMode", "rescue") 
  .option("cloudFiles.schemaLocation", "/Volumes/workspace/default/schemas/rescue_lab")
  .load(source_path))

(df.writeStream
  .option("checkpointLocation", checkpoint_path)
  .trigger(availableNow=True)
  .toTable("bronze_raw_data"))/Volumes/workspace/default/checkpoints/auto_loader/raw_data/commits/

In [0]:
  spark.sql("SHOW CATALOGS").display()